# Audit of Fake News Detection Model
### By: Jack Hariri and Noga Gottlieb
### New York University (Spring 2026)
### Auditing https://huggingface.co/hamzab/roberta-fake-news-classification 


In [1]:
#importing the necessary libraries and loading the pre-trained model

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import torch

tokenizer = AutoTokenizer.from_pretrained("hamzab/roberta-fake-news-classification")

model = AutoModelForSequenceClassification.from_pretrained("hamzab/roberta-fake-news-classification")

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: hamzab/roberta-fake-news-classification
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
#function to predict if the news is fake or real
def predict_fake(title,text):
    input_str = "<title>" + title + "<content>" +  text + "<end>"
    input_ids = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    device =  'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    with torch.no_grad():
        output = model(input_ids["input_ids"].to(device), attention_mask=input_ids["attention_mask"].to(device))
    return dict(zip(["Fake","Real"], [x.item() for x in list(torch.nn.Softmax()(output.logits)[0])] ))
    
print(predict_fake('This is a headline','This is real news, trust me'))

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

{'Fake': 0.999137282371521, 'Real': 0.0008626455091871321}


/opt/anaconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


In [ ]:
#loading the datasets
fake_news = pd.read_csv(
    "Fake.csv",
    engine="python",      
    quotechar='"',
    escapechar='\\',     
    on_bad_lines="skip"   
)
real_news = pd.read_csv(
    "True.csv",
    engine="python",   
    quotechar='"',
    escapechar='\\',      #
    on_bad_lines="skip"   
)

In [11]:
#creating a dictionary to store the results of the predictions
fake_news_results = {'fake': [],
                     'real': []}

fake_news_test = fake_news.head(10)
display(fake_news_test)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017"
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017"
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017"
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017"
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017"


In [ ]:
#predicting if the news is fake or real and storing the results in the dictionary
for _, article in fake_news_test.iterrows():
    result = predict_fake(article['title'], article['text'])
    fake_news_results['fake'].append(result['Fake'])
    fake_news_results['real'].append(result['Real'])
    


print (sum(fake_news_results['fake'])/len(fake_news_results['fake']))

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


0.7174275479730567


In [ ]:
#loading the test dataset columns
columns = [
    "id",
    "label",
    "statement",
    "subjects",
    "speaker",
    "job_title",
    "state_info",
    "party_affiliation",
    "barely_true_count",
    "false_count",
    "half_true_count",
    "mostly_true_count",
    "pants_on_fire_count",
    "context"
]

In [10]:
#loading the test dataset
newstotest = pd.read_csv(
    "test.tsv",
    sep="\t",
    header=None,     
    names=columns   
)

fake_news_test = newstotest.head(10)
display(fake_news_test)

,id,label,statement,subjects,speaker,job_title,state_info,party_affiliation,barely_true_count,false_count,half_true_count,mostly_true_count,pants_on_fire_count,context
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,42,23,18,Radio interview
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,0,0,0,a news conference
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,51,37,61,comments on ABC's This Week.
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,3,1,1,a radio show
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,2,2,7,a web video
5,5962.json,true,Over the past five years the federal governmen...,"federal-budget,pensions,retirement",brendan-doherty,NaN,Rhode Island,republican,1,2,1,1,0,a campaign website
6,7070.json,true,Says that Tennessee law requires that schools ...,"county-budget,county-government,education,taxes",stand-children-tennessee,Child and education advocacy organization.,Tennessee,none,0,0,0,0,0,in a post on Facebook.
7,1046.json,barely-true,"Says Vice President Joe Biden ""admits that the...","economy,stimulus",john-boehner,Speaker of the House of Representatives,Ohio,republican,13,22,11,4,2,a press release.
8,12849.json,true,Donald Trump is against marriage equality. He ...,"gays-and-lesbians,marriage",sean-patrick-maloney,Congressman for NY-18,New York,democrat,0,0,0,0,0,a speech at the Democratic National Convention
9,13270.json,barely-true,We know that more than half of Hillary Clinton...,foreign-policy,mike-pence,Governor,Indiana,republican,8,10,12,5,0,"comments on ""Meet the Press"""


In [8]:
for _, article in fake_news_test.iterrows():
    result = predict_fake(article['id'], article['statement'])
    fake_news_results['fake'].append(result['Fake'])
    fake_news_results['real'].append(result['Real'])
    print(f"Statement: {article['statement']}, Prediction: {result['Real']}, True Value: {article['label']}")
    


print (sum(fake_news_results['fake'])/len(fake_news_results['fake']))

Statement: Building a wall on the U.S.-Mexico border will take literally years., Prediction: 0.9999104738235474, True Value: true
Statement: Wisconsin is on pace to double the number of layoffs this year., Prediction: 0.9999163150787354, True Value: false
Statement: Says John McCain has done nothing to help the vets., Prediction: 0.0008580216090194881, True Value: false


/opt/anaconda3/lib/python3.13/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Statement: Suzanne Bonamici supports a plan that will cut choice for Medicare Advantage seniors., Prediction: 0.9996377229690552, True Value: half-true
Statement: When asked by a reporter whether hes at the center of a criminal scheme to violate campaign laws, Gov. Scott Walker nodded yes., Prediction: 0.520537257194519, True Value: pants-fire
Statement: Over the past five years the federal government has paid out $601 million in retirement and disability benefits to deceased former federal employees., Prediction: 0.9998948574066162, True Value: true
Statement: Says that Tennessee law requires that schools receive half of proceeds -- $31 million per year -- from a half-cent increase in the Shelby County sales tax., Prediction: 0.9995959401130676, True Value: true
Statement: Says Vice President Joe Biden "admits that the American people are being scammed" with the economic stimulus package., Prediction: 0.0002514163206797093, True Value: barely-true
Statement: Donald Trump is against ma

In [9]:
reuters_count = real_news['text'].str.contains('Reuters').sum()

print(f"Number of articles containing 'Reuters': {reuters_count}")

Number of articles containing 'Reuters': 21378
